# 🔍 Scout Agent — Draft

Scout is the first agent in the **Seesaw** pipeline. It takes a research question about AI safety / mechanistic interpretability and produces a structured **Research Plan** that the Lens agent will execute.

## Pipeline position

```
User
 │
 ▼
[Research Question]
 │
 ▼
┌─────────────────────────────────────┐
│              Scout                  │  ◄── YOU ARE HERE
│                                     │
│  search_arxiv()                     │
│  search_web()      ─── Firecrawl    │
│  scrape_url()      ─── Firecrawl    │
│  save_research_plan()               │
└─────────────────────────────────────┘
 │
 ▼
[Research Plan]  →  Lens  →  Quill  →  Report
```

## Stack
- **LangGraph** — agent loop and state management (`create_react_agent`)
- **Claude** — backbone LLM
- **arxiv** — academic paper search (no API key needed)
- **Firecrawl** — web search + deep URL scraping (mirrors Nova's pattern)

## 1. Setup

In [1]:
%pip install -q \
    langchain-anthropic \
    langgraph \
    arxiv \
    firecrawl-py \
    python-dotenv

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.8/232.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 13.0 MB/s eta 0:00:00


In [2]:
import os
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

import arxiv
from firecrawl import FirecrawlApp

load_dotenv()

print("✅ Imports OK")

/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


✅ Imports OK


In [ ]:
# ── API Keys ──────────────────────────────────────────────────────────────────
ANTHROPIC_API_KEY  = os.getenv("ANTHROPIC_API_KEY")
FIRECRAWL_API_KEY  = os.getenv("FIRECRAWL_API_KEY")

# ── Paths ─────────────────────────────────────────────────────────────────────
SCOUT_ROOT  = Path(".")
OUTPUTS_DIR = SCOUT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

# ── Status ────────────────────────────────────────────────────────────────────
print(f"Anthropic API key  : {'✅ set' if ANTHROPIC_API_KEY else '❌ missing — required'}")
print(f"Firecrawl API key  : {'✅ set' if FIRECRAWL_API_KEY else '❌ missing — required for search_web and scrape_url'}")
print(f"Outputs directory  : {OUTPUTS_DIR.resolve()}")

Anthropic API key  : ✅ set
Firecrawl API key  : ✅ set
Outputs directory  : /content/outputs


## 2. Scout Tools

Scout has five tools. Firecrawl handles all web I/O — consistent with Nova's pattern.

| Tool | Purpose | Backed by |
|---|---|---|
| `search_arxiv` | Search academic papers via the official arXiv API — structured, reliable | arxiv library (free) |
| `search_arxiv_web` | Search arxiv.org via Firecrawl — web-ranked, surfaces recent preprints | Firecrawl search |
| `search_web` | Search the broader web for blog posts, GitHub repos, Colab notebooks | Firecrawl search |
| `scrape_url` | Get full cleaned markdown from any URL | Firecrawl scrape |
| `save_research_plan` | Write the final Research Plan to disk | local file I/O |

**When to use which arxiv tool:**
- `search_arxiv` → structured keyword queries, known paper titles, getting clean abstracts
- `search_arxiv_web` → web-ranked results, recent preprints the API buries, finding related survey posts

In [5]:
@tool
def search_arxiv(query: str, max_results: int = 8) -> str:
    """Search arXiv for academic papers on mechanistic interpretability and AI safety.
    Use multiple targeted queries to cover different aspects of the research question.
    Best for finding foundational papers, circuit analysis, and formal methods.

    Args:
        query: The search query (e.g. 'GPT-2 indirect object identification circuit')
        max_results: Number of papers to return (default: 8)

    Returns:
        Formatted list of papers with titles, authors, abstracts, and URLs
    """
    client = arxiv.Client()
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance,
    )

    results = []
    for paper in client.results(search):
        authors = ", ".join(a.name for a in paper.authors[:3])
        if len(paper.authors) > 3:
            authors += " et al."
        results.append(
            f"**Title**: {paper.title}\n"
            f"**Authors**: {authors}\n"
            f"**Published**: {paper.published.strftime('%Y-%m-%d')}\n"
            f"**Abstract**: {paper.summary[:600]}...\n"
            f"**URL**: {paper.entry_id}\n"
        )

    if not results:
        return f"No papers found for query: '{query}'"

    return f"Found {len(results)} papers for '{query}':\n\n" + "\n---\n".join(results)


print("✅ search_arxiv defined")

✅ search_arxiv defined


In [6]:
@tool
def search_arxiv_web(query: str, limit: int = 5) -> str:
    """Search arxiv.org via Firecrawl for papers on mechanistic interpretability and AI safety.
    Complements search_arxiv: while search_arxiv uses the official API (structured, keyword-matched),
    this tool uses web search ranking — better for surfacing recent preprints, finding papers
    that cite a key work, or when the API's relevance ranking buries what you need.

    Args:
        query: The search query (e.g. 'transformer circuits attention head superposition')
        limit: Number of results to return (default: 5)

    Returns:
        Search results from arxiv.org with titles, URLs, and content snippets
    """
    if not FIRECRAWL_API_KEY:
        return "search_arxiv_web unavailable: FIRECRAWL_API_KEY not set."

    app = FirecrawlApp(api_key=FIRECRAWL_API_KEY)

    # Scope the search to arxiv.org
    arxiv_query = f"site:arxiv.org {query}"

    try:
        response = app.search(arxiv_query, limit=limit)

        raw_results = getattr(response, "data", None) or response
        if isinstance(raw_results, dict):
            raw_results = raw_results.get("data", [])

        if not raw_results:
            return f"No arxiv.org results found for: '{query}'"

        results = []
        for r in raw_results:
            if isinstance(r, dict):
                title   = r.get("title", "N/A")
                url     = r.get("url", "N/A")
                snippet = r.get("description") or r.get("markdown", "")
            else:
                title   = getattr(r, "title", "N/A")
                url     = getattr(r, "url", "N/A")
                snippet = getattr(r, "description", None) or getattr(r, "markdown", "") or ""

            # Only keep actual arxiv.org links
            if "arxiv.org" not in str(url):
                continue

            results.append(
                f"**Title**: {title}\n"
                f"**URL**: {url}\n"
                f"**Snippet**: {str(snippet)[:400]}...\n"
            )

        if not results:
            return f"No arxiv.org results found for: '{query}'"

        return f"arxiv.org web results for '{query}':\n\n" + "\n---\n".join(results)

    except Exception as e:
        return f"search_arxiv_web failed for '{query}': {str(e)}"


print("✅ search_arxiv_web defined")

✅ search_arxiv_web defined


In [7]:
@tool
def search_web(query: str, limit: int = 5) -> str:
    """Search the web for recent blog posts, GitHub repos, Colab notebooks,
    and community resources on AI safety and mechanistic interpretability.
    Returns URLs and content snippets. Use scrape_url to get full content
    from the most relevant results.

    Args:
        query: The search query
        limit: Number of results to return (default: 5)

    Returns:
        Search results with titles, URLs, and content snippets
    """
    if not FIRECRAWL_API_KEY:
        return "search_web unavailable: FIRECRAWL_API_KEY not set."

    app = FirecrawlApp(api_key=FIRECRAWL_API_KEY)

    try:
        response = app.search(query, limit=limit)

        # response.data is a list of SearchResult objects
        raw_results = getattr(response, "data", None) or response
        if isinstance(raw_results, dict):
            raw_results = raw_results.get("data", [])

        if not raw_results:
            return f"No web results found for: '{query}'"

        results = []
        for r in raw_results:
            # Handle both object and dict responses across SDK versions
            if isinstance(r, dict):
                title   = r.get("title", "N/A")
                url     = r.get("url", "N/A")
                snippet = r.get("description") or r.get("markdown", "")[:400]
            else:
                title   = getattr(r, "title", "N/A")
                url     = getattr(r, "url", "N/A")
                snippet = (getattr(r, "description", None) or getattr(r, "markdown", "") or "")[:400]

            results.append(
                f"**Title**: {title}\n"
                f"**URL**: {url}\n"
                f"**Snippet**: {snippet}...\n"
            )

        return f"Web results for '{query}':\n\n" + "\n---\n".join(results)

    except Exception as e:
        return f"search_web failed for '{query}': {str(e)}"


print("✅ search_web defined")

✅ search_web defined


In [8]:
# Firecrawl cache setting — mirrors Nova's scraping_handler.py
# maxAge=1 week gives ~500% faster scraping for static content (docs, papers, blogs)
_MAX_AGE_ONE_WEEK = 604_800_000  # milliseconds
_MAX_RETRIES      = 3
_MAX_CHARS        = 8_000        # truncate to keep context window manageable


@tool
def scrape_url(url: str) -> str:
    """Scrape and extract the full cleaned markdown content from a URL using Firecrawl.
    Use this after search_web to get the complete content of a specific page —
    full paper text, entire blog post, complete README, etc.

    Args:
        url: The URL to scrape

    Returns:
        Cleaned markdown content from the page (truncated at 8000 chars)
    """
    if not FIRECRAWL_API_KEY:
        return "scrape_url unavailable: FIRECRAWL_API_KEY not set."

    app = FirecrawlApp(api_key=FIRECRAWL_API_KEY)

    for attempt in range(_MAX_RETRIES):
        try:
            res = app.scrape_url(
                url,
                formats=["markdown"],
                # maxAge: use cached version if available — much faster
                # same strategy as Nova's scraping_handler.py
            )

            # Handle both object and dict responses across SDK versions
            if isinstance(res, dict):
                title    = res.get("metadata", {}).get("title", "N/A")
                markdown = res.get("markdown", "")
            else:
                title    = getattr(getattr(res, "metadata", None), "title", "N/A") or "N/A"
                markdown = getattr(res, "markdown", "") or ""

            if not markdown.strip():
                return f"No content returned for {url}"

            # Truncate to avoid blowing up context window
            if len(markdown) > _MAX_CHARS:
                markdown = markdown[:_MAX_CHARS] + "\n\n[... content truncated ...]"

            return f"# {title}\nSource: {url}\n\n{markdown}"

        except Exception as e:
            if attempt < _MAX_RETRIES - 1:
                import time
                delay = 5 * (2 ** attempt)  # exponential backoff: 5s, 10s
                print(f"  ⚠️ scrape attempt {attempt + 1} failed for {url} — retrying in {delay}s")
                time.sleep(delay)
            else:
                return f"Failed to scrape {url} after {_MAX_RETRIES} attempts: {str(e)}"

    return f"Failed to scrape {url}"


print("✅ scrape_url defined")

✅ scrape_url defined


In [9]:
@tool
def save_research_plan(plan: str, filename: str = "research_plan.md") -> str:
    """Save the final structured Research Plan to disk.
    Call this ONLY when you have gathered sufficient information and are ready
    to produce the final output. This signals the end of the Scout workflow.

    Args:
        plan: The complete research plan in markdown format
        filename: Output filename (default: research_plan.md)

    Returns:
        Confirmation message with the saved file path
    """
    output_path = OUTPUTS_DIR / filename
    output_path.write_text(plan, encoding="utf-8")
    return (
        f"✅ Research plan saved to: {output_path.resolve()}\n\n"
        f"Plan preview (first 500 chars):\n{plan[:500]}..."
    )


print("✅ save_research_plan defined")

✅ save_research_plan defined


In [10]:
# All tools registered in one place — easy to extend later
SCOUT_TOOLS = [
    search_arxiv,
    search_arxiv_web,
    search_web,
    scrape_url,
    save_research_plan,
]

print(f"✅ {len(SCOUT_TOOLS)} tools registered: {[t.name for t in SCOUT_TOOLS]}")

✅ 5 tools registered: ['search_arxiv', 'search_arxiv_web', 'search_web', 'scrape_url', 'save_research_plan']


In [12]:
SCOUT_SYSTEM_PROMPT = """
You are Scout, an AI safety research agent specialised in mechanistic interpretability.

Your job: take a research question and produce a comprehensive Research Plan by gathering
information from academic papers, blog posts, GitHub repositories, and documentation.
The Research Plan will be handed to the Lens agent, which will run the experiments you specify.

## Workflow

Follow these steps in order:

1. **Decompose** — break the research question into 3-5 sub-topics
2. **Search arXiv** — run at least 3 targeted searches with different queries
3. **Search the web** — find blog posts, GitHub repos, and Colab notebooks via search_web
4. **Scrape key URLs** — use scrape_url to get the full content of the 2-3 most relevant pages
5. **Synthesise** — identify patterns, hypotheses, and open questions
6. **Save** — call `save_research_plan` with the final structured plan

## Research Plan Format

Your final output passed to `save_research_plan` MUST follow this exact structure:

```
# Research Plan: <research_question>

**Date**: <date>
**Status**: Draft

---

## Background
<2-3 paragraphs summarising the current state of knowledge on this topic>

## Key Papers & Resources
<bullet list — title, authors, year, one-line description, URL>

## Hypotheses
<numbered list of specific, testable hypotheses>

## Proposed Experiments
<For each experiment, use this exact format:>

### Experiment N: <name>
- **Lens Tool**: <one of: logit_lens | attention_pattern | ablation | activation_patching | direct_logit_attribution>
- **Model**: <e.g. gpt2-small, pythia-160m>
- **Dataset / Prompts**: <what inputs to use>
- **What to measure**: <specific metric or observation>
- **Hypothesis tested**: <which hypothesis above this tests>
- **Expected outcome**: <what you predict>

## Target Models
<which models and why — focus on TransformerLens-compatible small models>

## Expected Findings
<what the experiments are likely to reveal>

## Open Questions
<what remains unknown after this research plan is executed>
```

## Available Lens Tools (for Proposed Experiments)

| Tool | What it does |
|---|---|
| `logit_lens` | Visualises how predictions evolve across layers |
| `attention_pattern` | Shows which tokens attend to which at each layer |
| `ablation` | Zero/mean ablates heads or MLPs to measure their importance |
| `activation_patching` | Patches activations to find causal components |
| `direct_logit_attribution` | Decomposes final logit output by layer and component |

## Guidelines
- Focus on mechanistic interpretability and AI safety implications
- Prioritise papers from 2022 onwards (the modern mech interp era)
- Be concrete about experiments — Lens will run exactly what you specify
- Each hypothesis must be testable with TransformerLens on small models
- Prefer GPT-2 Small and Pythia models for experiments (fast, well-studied)
""".strip()

## 4. Build Scout Agent with LangGraph

We use `create_react_agent` — a prebuilt ReAct loop that:
1. Calls Claude with the available tools
2. If Claude calls a tool → executes it → feeds result back
3. Repeats until Claude produces a final answer

This mirrors Nova's `handle_agent_loop` but uses LangGraph's graph infrastructure instead of a manual `while True` loop.

In [17]:
# ── Model ─────────────────────────────────────────────────────────────────────
# claude-opus-4-5 for best research planning quality
# swap to claude-sonnet-4-5 for faster / cheaper runs
model = ChatAnthropic(
    model="claude-opus-4-5",
    api_key=ANTHROPIC_API_KEY,
    temperature=1,
    max_tokens=8_000,
)

# ── Memory ────────────────────────────────────────────────────────────────────
# MemorySaver keeps the full message history in RAM across streamed chunks
memory = MemorySaver()

# ── Agent ─────────────────────────────────────────────────────────────────────
scout = create_react_agent(
    model=model,
    tools=SCOUT_TOOLS,
    checkpointer=memory,
    prompt=SystemMessage(content=SCOUT_SYSTEM_PROMPT),
)

print("✅ Scout agent ready")
print(f"   Model  : claude-opus-4-5")
print(f"   Tools  : {[t.name for t in SCOUT_TOOLS]}")
print(f"   Memory : MemorySaver (in-process)")

✅ Scout agent ready
   Model  : claude-opus-4-5
   Tools  : ['search_arxiv', 'search_arxiv_web', 'search_web', 'scrape_url', 'save_research_plan']
   Memory : MemorySaver (in-process)


/tmp/ipykernel_803/3022755526.py:16: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  scout = create_react_agent(


## 5. Run Scout

Edit `RESEARCH_QUESTION` below and run the cells. Scout will:
- Search arXiv and the web
- Scrape the most relevant URLs via Firecrawl
- Save a `research_plan.md` to the `outputs/` folder

In [18]:
# ── Research Question ─────────────────────────────────────────────────────────
# Classic mech interp task — well-studied, good for validating the pipeline
RESEARCH_QUESTION = (
    "How does GPT-2 Small implement the Indirect Object Identification (IOI) task? "
    "Which attention heads and circuits are responsible, and what are the key "
    "computational mechanisms?"
)

# Thread ID — change this to start a fresh session
THREAD_ID = f"scout-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
config = {"configurable": {"thread_id": THREAD_ID}}

print(f"🔍 Research Question:")
print(f"   {RESEARCH_QUESTION}")
print(f"\n   Thread : {THREAD_ID}")
print("\n" + "=" * 70)
print("Running Scout...\n")

🔍 Research Question:
   How does GPT-2 Small implement the Indirect Object Identification (IOI) task? Which attention heads and circuits are responsible, and what are the key computational mechanisms?

   Thread : scout-20260510-100906

Running Scout...



In [19]:
# ── Stream the agent loop ─────────────────────────────────────────────────────
# stream_mode="updates" yields one dict per graph node update

for chunk in scout.stream(
    {"messages": [HumanMessage(content=RESEARCH_QUESTION)]},
    config=config,
    stream_mode="updates",
):
    for node_name, node_update in chunk.items():
        messages = node_update.get("messages", [])

        for msg in messages:
            # ── Tool call (agent deciding to use a tool) ───────────────────
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    args_preview = str(tc.get("args", {}))[:120]
                    print(f"🔧 [{tc['name']}] {args_preview}")

            # ── Tool result (result coming back from the tool) ─────────────
            elif hasattr(msg, "name") and msg.name in [t.name for t in SCOUT_TOOLS]:
                content_preview = str(msg.content)[:200].replace("\n", " ")
                print(f"   ↳ result: {content_preview}...")

            # ── Agent reasoning / final text ───────────────────────────────
            elif hasattr(msg, "content") and msg.content:
                content = msg.content
                if isinstance(content, list):
                    for block in content:
                        if isinstance(block, dict) and block.get("type") == "text":
                            preview = block["text"][:300].replace("\n", " ")
                            print(f"\n🤖 Scout: {preview}...\n")
                elif isinstance(content, str) and content.strip():
                    preview = content[:300].replace("\n", " ")
                    print(f"\n🤖 Scout: {preview}...\n")

print("\n" + "=" * 70)
print("✅ Scout run complete")

🔧 [search_arxiv] {'query': 'GPT-2 indirect object identification circuit interpretability', 'max_results': 10}
🔧 [search_arxiv] {'query': 'transformer attention head circuit mechanistic interpretability', 'max_results': 8}
🔧 [search_arxiv] {'query': 'induction heads copying behavior language models', 'max_results': 8}
   ↳ result: Found 8 papers for 'induction heads copying behavior language models':  **Title**: A Property Induction Framework for Neural Language Models **Authors**: Kanishka Misra, Julia Taylor Rayz, Allyson Ett...
   ↳ result: Found 10 papers for 'GPT-2 indirect object identification circuit interpretability':  **Title**: Interpretability in the Wild: a Circuit for Indirect Object Identification in GPT-2 small **Authors**: ...
   ↳ result: Found 8 papers for 'transformer attention head circuit mechanistic interpretability':  **Title**: nnterp: A Standardized Interface for Mechanistic Interpretability of Transformers **Authors**: Clément...
🔧 [search_arxiv] {'query': 'a

## 6. Display Research Plan

In [22]:
# from IPython.display import Markdown, display

# plan_path = OUTPUTS_DIR / "research_plan.md"

# if plan_path.exists():
#     plan_text = plan_path.read_text(encoding="utf-8")
#     print(f"📄 Research plan: {plan_path.resolve()}")
#     print(f"   Size: {len(plan_text):,} chars\n")
#     display(Markdown(plan_text))
# else:
#     print("⚠️  No research_plan.md found in outputs/.")
#     print("    Scout may not have called save_research_plan yet.")
#     print("    Check the streaming output above for errors.")


from IPython.display import Markdown, display
from pathlib import Path

# Search for any file ending with research_plan (any extension)
matches = list(OUTPUTS_DIR.glob("*research_plan*"))

if matches:
    plan_path = matches[0]  # use the first match
    plan_text = plan_path.read_text(encoding="utf-8")
    print(f"📄 Research plan: {plan_path.resolve()}")
    print(f"   Size: {len(plan_text):,} chars\n")
    display(Markdown(plan_text))
else:
    print("⚠️  No research_plan file found in outputs/.")
    print("    Scout may not have called save_research_plan yet.")
    print("    Check the streaming output above for errors.")

📄 Research plan: /content/outputs/ioi_circuit_research_plan.md
   Size: 17,289 chars



# Research Plan: Indirect Object Identification (IOI) Circuit in GPT-2 Small

**Date**: 2025-01-14
**Status**: Draft

---

## Background

The Indirect Object Identification (IOI) task is a canonical benchmark for mechanistic interpretability research, first comprehensively studied by Wang et al. (2022). The task involves sentences like "When Mary and John went to the store, John gave a drink to" where the model must predict "Mary" (the indirect object) rather than "John" (the subject of the final clause). This requires the model to track which name appeared first, which name appeared second, and suppress the repeated name. The task elegantly probes coreference resolution and name tracking capabilities that are fundamental to language understanding.

Wang et al.'s groundbreaking work reverse-engineered the IOI circuit in GPT-2 Small, identifying 26 attention heads organized into 7 functional classes that implement the task through a structured computational pipeline. The circuit includes: (1) **Previous Token Heads** that attend to the immediately preceding token, (2) **Duplicate Token Heads** that detect when a token has appeared before, (3) **S-Inhibition Heads** that inhibit the repeated subject, (4) **Induction Heads** that perform sequence completion, (5) **Name Mover Heads** that copy the correct name to the output, (6) **Backup Name Mover Heads** that provide redundancy, and (7) **Negative Name Mover Heads** that oppose incorrect predictions.

This circuit exhibits several fascinating properties relevant to AI safety and robustness. First, the circuit shows **backup behavior** - when key components are ablated, backup heads compensate, suggesting learned redundancy. Second, the heads exhibit functional specialization with composable behaviors - early heads detect patterns that later heads use for decisions. Third, the discovery methods (activation patching, path patching, direct logit attribution) have become standard techniques in the mechanistic interpretability toolkit. Understanding how this circuit works provides a template for understanding more complex behaviors in larger models.

## Key Papers & Resources

- **Interpretability in the Wild: a Circuit for Indirect Object Identification in GPT-2 Small** - Wang, Variengien, Conmy et al., 2022 - The foundational paper reverse-engineering the full IOI circuit with 26 heads in 7 classes. [arXiv:2211.00593](http://arxiv.org/abs/2211.00593)

- **In-context Learning and Induction Heads** - Olsson, Elhage, Nanda et al., 2022 - Describes induction heads that implement [A][B]...[A] → [B] completion, a key building block for IOI. [arXiv:2209.11895](http://arxiv.org/abs/2209.11895)

- **Copy Suppression: Comprehensively Understanding an Attention Head** - McDougall, Conmy, Rushing et al., 2023 - Deep dive into L10H7, a negative head that suppresses naive copying behavior. [arXiv:2310.04625](http://arxiv.org/abs/2310.04625)

- **How to Use and Interpret Activation Patching** - Heimersheim & Nanda, 2024 - Best practices for activation patching, the primary technique used in IOI circuit discovery. [arXiv:2404.15255](http://arxiv.org/abs/2404.15255)

- **Towards Automated Circuit Discovery for Mechanistic Interpretability (ACDC)** - Conmy, Mavor-Parker, Lynch et al., 2023 - Systematizes circuit discovery and introduces automated methods. [arXiv:2304.14997](http://arxiv.org/abs/2304.14997)

- **Attribution Patching Outperforms Automated Circuit Discovery** - Syed, Rager, Conmy, 2023 - Shows linear approximations to activation patching can efficiently identify circuits. [arXiv:2310.10348](http://arxiv.org/abs/2310.10348)

- **An Adversarial Example for Direct Logit Attribution: Memory Management in GELU-4L** - Janiak et al., 2023 - Demonstrates potential pitfalls of DLA when erasure heads are present. [arXiv:2310.07325](http://arxiv.org/abs/2310.07325)

- **Adaptive Circuit Behavior and Generalization in Mechanistic Interpretability** - Nainani et al., 2024 - Studies how circuits generalize across prompt formats. [arXiv:2411.16105](http://arxiv.org/abs/2411.16105)

- **Emergence of Minimal Circuits for Indirect Object Identification** - Adhikari, 2025 - Shows minimal 2-head attention-only models can solve symbolic IOI. [arXiv:2510.25013](http://arxiv.org/abs/2510.25013)

## Hypotheses

1. **Name Mover Heads (L9H9, L9H6, L10H0) will show strong positive direct logit attribution toward the indirect object token (IO)**, directly copying its name to the output position. These heads should attend primarily from the final token position to the IO token position.

2. **S-Inhibition Heads (L7H3, L7H9, L8H6, L8H10) will suppress attention to the repeated subject (S2)** by writing negative information about S2 into the residual stream, which Name Mover Heads then read. This creates a "don't copy this name" signal.

3. **Duplicate Token Heads (L0H1, L0H10, L3H0) will attend from the second occurrence of a name (S2) to its first occurrence (S1)**, detecting that the name has appeared before. This detection enables downstream S-Inhibition.

4. **Induction Heads will show the characteristic [A][B]...[A] → [B] attention pattern**, attending from the query position to tokens that followed previous occurrences of the current token, providing another mechanism for name tracking.

5. **Ablating Name Mover Heads will significantly reduce IOI performance, but Backup Name Movers (L9H0, L9H7, L10H1, L10H2, L10H6, L10H10, L11H2, L11H9) will partially compensate**, demonstrating learned redundancy in the circuit.

6. **The circuit will show compositional structure**: early heads (Duplicate Token, Previous Token) provide features that middle-layer heads (S-Inhibition, Induction) process, which then feed into late-layer heads (Name Movers). Ablating early heads should cascade and affect downstream components.

7. **Negative/Copy Suppression Head L10H7 will show negative direct logit attribution for the subject token**, actively suppressing the "wrong" answer to improve calibration.

8. **The logit lens will show IOI predictions emerging gradually across layers**, with the correct name (IO) becoming increasingly predicted in later layers as the circuit executes its computation.

## Proposed Experiments

### Experiment 1: Verify Name Mover Head Attention Patterns
- **Lens Tool**: attention_pattern
- **Model**: gpt2-small
- **Dataset / Prompts**: Standard IOI prompts with varied names, e.g., "When Mary and John went to the store, John gave a drink to" (expect "Mary"). Use ABBA and BABA templates: ABBA = "When [IO] and [S] went to the store, [S] gave a drink to"; BABA = "When [S] and [IO] went to the store, [S] gave a drink to". Generate 50 prompts per template with random name pairs.
- **What to measure**: Attention weights from the final (prediction) token position to all name positions. Specifically examine heads L9H9, L9H6, L10H0 (known Name Movers).
- **Hypothesis tested**: Hypothesis 1
- **Expected outcome**: Name Mover Heads should show >50% attention weight to the IO token position from the final position, with minimal attention to S2.

### Experiment 2: Direct Logit Attribution of Name Mover Heads
- **Lens Tool**: direct_logit_attribution
- **Model**: gpt2-small
- **Dataset / Prompts**: Same IOI dataset as Experiment 1 (50 ABBA + 50 BABA prompts).
- **What to measure**: Direct contribution of each attention head's output to the logit difference (logit[IO] - logit[S]). Focus on Name Mover Heads (L9H9, L9H6, L10H0) and Negative Head (L10H7).
- **Hypothesis tested**: Hypotheses 1 and 7
- **Expected outcome**: Name Mover Heads should have large positive DLA (>2.0 logits contribution on average). L10H7 should have negative DLA, suppressing the subject token.

### Experiment 3: S-Inhibition Head Attention to Duplicate Names
- **Lens Tool**: attention_pattern
- **Model**: gpt2-small
- **Dataset / Prompts**: IOI prompts where S1 (first occurrence of subject) and S2 (second occurrence) are at known positions. 50 prompts with clear positional structure.
- **What to measure**: Attention weights of S-Inhibition heads (L7H3, L7H9, L8H6, L8H10) from the final position to S1 and S2 positions.
- **Hypothesis tested**: Hypothesis 2
- **Expected outcome**: S-Inhibition heads should attend more strongly to S2 (the repeated name) than to IO, enabling them to write "don't copy this" information.

### Experiment 4: Duplicate Token Head Detection Pattern
- **Lens Tool**: attention_pattern
- **Model**: gpt2-small
- **Dataset / Prompts**: IOI prompts where we track attention from S2's position. 50 ABBA and 50 BABA prompts.
- **What to measure**: Attention patterns of Duplicate Token Heads (L0H1, L0H10, L3H0) specifically from the S2 token position.
- **Hypothesis tested**: Hypothesis 3
- **Expected outcome**: These heads should attend from S2's position back to S1's position (the first occurrence of the same name), showing >40% attention weight to the matching earlier token.

### Experiment 5: Name Mover Head Ablation and Backup Compensation
- **Lens Tool**: ablation
- **Model**: gpt2-small
- **Dataset / Prompts**: 100 IOI prompts (50 ABBA + 50 BABA).
- **What to measure**: 
  1. Baseline logit difference (logit[IO] - logit[S])
  2. Logit difference after mean-ablating primary Name Mover Heads (L9H9, L9H6, L10H0)
  3. Logit difference after ablating both primary and backup Name Movers
- **Hypothesis tested**: Hypothesis 5
- **Expected outcome**: Ablating primary Name Movers should reduce logit difference by ~40-60%, but not to zero. Ablating all Name Movers (primary + backup) should nearly eliminate positive logit difference, demonstrating backup compensation.

### Experiment 6: Cascade Effects from Ablating Early Heads
- **Lens Tool**: ablation
- **Model**: gpt2-small
- **Dataset / Prompts**: 100 IOI prompts.
- **What to measure**: 
  1. Baseline accuracy on IOI
  2. Accuracy after ablating Duplicate Token Heads (L0H1, L0H10, L3H0)
  3. Downstream attention patterns of S-Inhibition heads after Duplicate Token ablation
- **Hypothesis tested**: Hypothesis 6
- **Expected outcome**: Ablating Duplicate Token Heads should reduce IOI accuracy significantly (>20% drop) and should alter S-Inhibition head attention patterns, demonstrating compositional dependence.

### Experiment 7: Activation Patching to Identify Causal Components
- **Lens Tool**: activation_patching
- **Model**: gpt2-small
- **Dataset / Prompts**: Paired prompts where we patch from a "corrupted" prompt (names swapped) to the original. E.g., Original: "When Mary and John went... John gave to" → "Mary"; Corrupted: "When John and Mary went... Mary gave to" → "John". 50 prompt pairs.
- **What to measure**: Recovery of correct IOI behavior when patching each attention head's output from corrupted → clean run, measured as fraction of logit difference recovered.
- **Hypothesis tested**: Hypotheses 1, 2, 3, 6
- **Expected outcome**: Name Mover Heads should show highest recovery (>30% each). S-Inhibition and Duplicate Token Heads should show moderate recovery (10-20% each). Random heads should show <5% recovery.

### Experiment 8: Logit Lens Tracking Prediction Evolution
- **Lens Tool**: logit_lens
- **Model**: gpt2-small
- **Dataset / Prompts**: 50 IOI prompts, examining the residual stream at each layer at the final token position.
- **What to measure**: Probability assigned to IO vs S at each layer, tracking when the correct prediction emerges.
- **Hypothesis tested**: Hypothesis 8
- **Expected outcome**: Early layers (0-5) should show relatively flat or slightly S-favoring predictions. Middle layers (6-8) should show transition as S-Inhibition takes effect. Late layers (9-11) should strongly favor IO, showing the Name Mover contribution.

### Experiment 9: Induction Head Pattern Verification
- **Lens Tool**: attention_pattern
- **Model**: gpt2-small
- **Dataset / Prompts**: IOI prompts plus simpler induction-testing prompts of form "[A][B]...[A]" to verify the basic pattern. 30 IOI prompts + 20 pure induction prompts.
- **What to measure**: Attention patterns of known induction heads (L5H5, L6H9) looking for the characteristic "attend to token after previous occurrence" pattern.
- **Hypothesis tested**: Hypothesis 4
- **Expected outcome**: Induction heads should show strong attention from the position after [A]'s second occurrence to the position of [B] (the token that followed [A]'s first occurrence), with attention weight >30%.

### Experiment 10: Template Generalization Test
- **Lens Tool**: attention_pattern
- **Model**: gpt2-small  
- **Dataset / Prompts**: Multiple IOI templates beyond ABBA/BABA:
  - Template 1 (ABBA): "When [IO] and [S] went to the store, [S] gave a drink to"
  - Template 2 (BABA): "When [S] and [IO] went to the store, [S] gave a drink to"
  - Template 3: "Then [IO] and [S] were working at the restaurant. [S] gave a menu to"
  - Template 4: "[IO] and [S] went to the park. [S] handed a frisbee to"
  20 prompts per template (80 total).
- **What to measure**: Consistency of Name Mover attention patterns across templates.
- **Hypothesis tested**: Hypothesis 1 (generalization)
- **Expected outcome**: Name Mover heads should consistently attend to IO across all templates, with <15% variance in attention weight across templates, demonstrating robust circuit behavior.

## Target Models

**Primary Model: GPT-2 Small (124M parameters)**
- The IOI circuit was originally discovered in this model
- 12 layers, 12 heads per layer, 768-dimensional residual stream
- Well-documented circuit with known head functions
- Fast inference allowing comprehensive ablation studies
- Native TransformerLens support with verified numerical accuracy

**Secondary Model: Pythia-160M**
- Similar scale to GPT-2 Small but from a different training distribution
- Allows testing whether IOI circuits generalize across model families
- TransformerLens compatible
- Can verify whether the same functional head classes exist

**Potential Extension: GPT-2 Medium (355M)**
- If time permits, can verify whether the circuit scales/changes with model size
- May reveal additional backup heads or circuit refinements

## Expected Findings

1. **Confirmation of the 7-class head taxonomy**: Experiments should validate that the functional classifications (Name Movers, S-Inhibition, Duplicate Token, etc.) accurately describe head behavior in terms of both attention patterns and causal importance.

2. **Quantitative circuit importance ranking**: Direct logit attribution should show that Name Mover Heads (especially L9H9) contribute the largest positive signal, while L10H7 contributes the largest negative signal. This ranking should be consistent across prompt templates.

3. **Compositional structure verification**: Ablation cascades should demonstrate that the circuit has a clear information flow: Duplicate Token Heads → S-Inhibition Heads → Name Mover Heads, with each layer depending on computations from earlier layers.

4. **Backup head compensation**: When primary Name Movers are ablated, performance should degrade gracefully (not catastrophically), with backup heads partially compensating. This suggests learned redundancy for this important capability.

5. **Layer-wise prediction dynamics**: Logit lens should reveal that the model "decides" the correct answer primarily in layers 9-11, with middle layers (7-8) showing the critical transition where S-Inhibition takes effect.

6. **Template robustness**: The circuit should show consistent behavior across different surface-form templates, suggesting it implements a genuine algorithm for coreference tracking rather than memorized patterns.

## Open Questions

1. **What determines backup head allocation?** Why are there more backup heads for some functions than others? Is this related to training dynamics or task frequency?

2. **How did this circuit emerge during training?** The training dynamics that give rise to these specialized, composable components are not well understood. Did Name Movers emerge first, or did the detection heads (Duplicate Token) develop earlier?

3. **Does the circuit generalize to more complex coreference scenarios?** The IOI task involves only two names and simple structure. How does the model handle 3+ names, nested clauses, or longer-range dependencies?

4. **What is the role of MLP layers in the circuit?** The IOI paper focused primarily on attention heads. MLP layers may play important supporting roles (e.g., in name representation or output formatting) that remain underexplored.

5. **How does superposition affect the circuit?** Names and name-related features may be represented in superposition. Understanding how the circuit operates in this regime could reveal limitations and failure modes.

6. **Can we find adversarial examples that break the circuit?** Are there prompt constructions that exploit the circuit's specific structure to cause systematic failures?

7. **Is there cross-model universality?** Do models trained differently (different architectures, data, objectives) develop similar functional circuits for IOI, or are there multiple algorithmic solutions?

8. **How does the Copy Suppression head (L10H7) interact with the IOI circuit in edge cases?** When its suppression is helpful vs. harmful, and how does it balance with Name Mover contributions?


## 7. Inspect Agent State (Debug)

LangGraph's checkpointer stores the full message history. Useful for debugging what the agent saw.

In [23]:
state    = scout.get_state(config)
messages = state.values.get("messages", [])

print(f"Total messages in thread: {len(messages)}\n")

for i, msg in enumerate(messages):
    role      = msg.__class__.__name__.replace("Message", "")
    tool_name = getattr(msg, "name", "")
    label     = f"[{role}]" if not tool_name else f"[Tool: {tool_name}]"

    content = msg.content
    if isinstance(content, list):
        preview = " | ".join(
            b.get("text", "")[:80] if isinstance(b, dict) else str(b)[:80]
            for b in content
        )
    else:
        preview = str(content)[:120].replace("\n", " ")

    print(f"{i:02d}  {label:<30} {preview}")

Total messages in thread: 41

00  [Human]                        How does GPT-2 Small implement the Indirect Object Identification (IOI) task? Which attention heads and circuits are res
01  [AI]                           I'll investigate how GPT-2 Small implements the Indirect Object Identification ( |  |  | 
02  [Tool: search_arxiv]           Found 10 papers for 'GPT-2 indirect object identification circuit interpretability':  **Title**: Interpretability in the
03  [Tool: search_arxiv]           Found 8 papers for 'transformer attention head circuit mechanistic interpretability':  **Title**: nnterp: A Standardized
04  [Tool: search_arxiv]           Found 8 papers for 'induction heads copying behavior language models':  **Title**: A Property Induction Framework for Ne
05  [AI]                           Excellent! I found the key paper by Kevin Wang et al. (2022). Let me search for  |  |  | 
06  [Tool: search_arxiv]           Found 6 papers for 'activation patching causal intervention t

## 8. Next Steps

This notebook is a **draft** of Scout. Before wiring into the full Seesaw pipeline:

**Short term**
- [ ] Move tools into `scout/mcp_server/src/tools/` and `app/` following Nova's pattern
- [ ] Add `config/settings.py` — model selection, API key management (mirror Nova's `Settings` class)
- [ ] Add `config/constants.py` — file/folder name constants
- [ ] Add `config/prompts.py` — move system prompt here as a constant
- [ ] Add HITL checkpoint: pause after plan is saved, ask human to approve before Lens runs

**Tools to add**
- [ ] `search_semantic_scholar` — Semantic Scholar API (free, great AI coverage)
- [ ] `search_lesswrong` — LessWrong / Alignment Forum posts
- [ ] `process_github_urls` — gitingest for repos (port directly from Nova)
- [ ] `transcribe_youtube` — for recorded talks (port directly from Nova)

**Orchestrator integration**
- [ ] Expose as an MCP server (`scout/mcp_server/src/server.py`)
- [ ] Connect via `orchestrator/src/clients/scout_client.py`
- [ ] Wire HITL gate between Scout output and Lens input